<a href="https://colab.research.google.com/github/Kemal1101/Model-Comparison-for-Sistem-Tes-Minat-Career/blob/main/Preprcessing_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
RIASEC Data Preprocessing
Deskripsi: Kode ini melakukan pembersihan data, standarisasi kolom,
           feature engineering (skor RIASEC), dan ekspor data hasil kuesioner minat karir.
"""

import pandas as pd
import numpy as np
from google.colab import files
from google.colab import drive

# --- TAHAP 0: Inisialisasi ---
# Menghubungkan Google Colab dengan Google Drive untuk akses file
drive.mount('/content/drive')

# --- TAHAP 1: Memuat Dataset ---
# Menentukan path file dataset CSV di Google Drive
file_path = "/content/drive/MyDrive/Kuis_SBP/Survey Tes Penentuan Minat Karir Berbasis RIASEC (Responses) - Form Responses 1.csv"
df = pd.read_csv(file_path)

# --- TAHAP 2: Standarisasi Nama Kolom ---
# Mengubah pertanyaan panjang dari kuesioner menjadi kode singkat (R1-R5, I1-I5, dsb.)
riasec_cols = []
categories = ['R', 'I', 'A', 'S', 'E', 'C']
for cat in categories:
    for i in range(1, 6):
        riasec_cols.append(f'{cat}{i}')

# Mapping: Kolom indeks ke-5 hingga ke-34 diubah namanya sesuai urutan riasec_cols
rename_dict = {df.columns[i+5]: riasec_cols[i] for i in range(30)}
df_preprocessed = df.rename(columns=rename_dict)

# --- TAHAP 3: Data Cleaning (Pembersihan Data) ---
# 1. Menghapus kolom 'Timestamp' karena tidak digunakan dalam analisis statistik
df_preprocessed = df_preprocessed.drop(columns=['Timestamp'])

# 2. Menghapus whitespace (spasi di awal/akhir) pada kolom teks penting
text_cols = ['Nama Lengkap', 'Jenis Kelamin', 'Instansi Pendidikan', 'Program Studi']
for col in text_cols:
    df_preprocessed[col] = df_preprocessed[col].str.strip()

# 3. Normalisasi teks ke format Title Case (Huruf besar di awal kata)
df_preprocessed['Instansi Pendidikan'] = df_preprocessed['Instansi Pendidikan'].str.title()
df_preprocessed['Program Studi'] = df_preprocessed['Program Studi'].str.title()

# --- TAHAP 4: Encoding Data ---
# Mengubah data kategorikal 'Jenis Kelamin' menjadi numerik untuk kebutuhan machine learning/analisis
df_preprocessed['Jenis Kelamin_Encoded'] = df_preprocessed['Jenis Kelamin'].map({'Laki-laki': 1, 'Perempuan': 0})

# --- TAHAP 5: Feature Engineering ---
# 1. Menghitung rata-rata skor untuk setiap kategori (Realistic, Investigative, dll.)
for cat in categories:
    cat_cols = [f'{cat}{i}' for i in range(1, 6)]
    df_preprocessed[f'Skor_{cat}'] = df_preprocessed[cat_cols].mean(axis=1)

# 2. Menentukan 'Kategori Dominan' berdasarkan skor tertinggi di antara 6 kategori
score_cols = [f'Skor_{cat}' for cat in categories]
cat_names = {
    'Skor_R': 'Realistic', 'Skor_I': 'Investigative', 'Skor_A': 'Artistic',
    'Skor_S': 'Social', 'Skor_E': 'Enterprising', 'Skor_C': 'Conventional'
}
df_preprocessed['Kategori_Dominan'] = df_preprocessed[score_cols].idxmax(axis=1).map(cat_names)

# --- TAHAP 6: Reorganisasi Kolom ---
# Menyusun ulang kolom agar lebih mudah dibaca (Informasi Identitas -> Jawaban -> Hasil Skor)
front_cols = ['Nama Lengkap', 'Jenis Kelamin', 'Jenis Kelamin_Encoded', 'Instansi Pendidikan', 'Program Studi']
feature_cols = riasec_cols
result_cols = score_cols + ['Kategori_Dominan']

df_final = df_preprocessed[front_cols + feature_cols + result_cols]

# --- TAHAP 7: Export & Download ---
output_filename = "Preprocessed_RIASEC_Dataset.csv"
df_final.to_csv(output_filename, index=False)

print("Preprocessing selesai!")
print(f"Mengunduh file: {output_filename} ...")

# Memicu download file otomatis ke perangkat lokal user
files.download(output_filename)


Mounted at /content/drive
Preprocessing selesai!
Mengunduh file: Preprocessed_RIASEC_Dataset.csv ...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>